In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import yaml

sys.path.insert(0, str(Path().absolute().parent.parent.parent))

In [ ]:
from test.lib.files_utils import SimulationOutput, LisLoader, ISOTOPES, func

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
PART = 32
input_path = next((Path().resolve().parent / 'data' / 'benchmark' / 'results' / 'outputs_A100-3' / f'20111116_20111212_103_k0_{PART}').glob('*.yaml'))
outs = SimulationOutput.from_yaml(yaml.load(input_path.read_text(), Loader=yaml.CLoader))

In [ ]:
input_path = next((Path().resolve().parent / 'data' / 'uncertainty' / 'outputs' / f'20111116_20111212_1024').glob('*.yaml'))
outs = SimulationOutput.from_yaml(yaml.load(input_path.read_text(), Loader=yaml.CLoader))

In [ ]:
input_path = next((Path().resolve().parent / 'uncertainty' / 'tmp').glob('*.yaml'))
outs = SimulationOutput.from_yaml(yaml.load(input_path.read_text(), Loader=yaml.CLoader))
print(input_path)

In [ ]:
lis_loader = LisLoader(Path().absolute().parent / 'data' / 'LIS_Default2020_Proton')
rows = []
fluxes = outs.modulate(lis_loader)
for i, (out, flux) in enumerate(zip(outs, fluxes)):
    for j, (r, f) in enumerate(zip(*flux.rig_flux)):
        vars, evs, nbins = 0, 0, 0
        for iso, ot in out.items():
            lis = lis_loader[iso]
            in_rig, out_rig, out_dist, n_part = ot.input_rig[j:j + 1], ot.output_rig[j], ot.output_dist[j], \
                ot.n_particles[j]

            lis_flux_en_out = func.lin_log_interpolation(lis.energy.to_rigidity(iso), lis.flux, out_rig)

            A, B, N = (lis_flux_en_out / out_rig ** 2), out_dist, n_part
            C = func.d_rig_to_en(in_rig.to_energy(iso), in_rig, iso.Z, iso.A)
            vars += C ** 2 * (np.sum(B * A ** 2) - np.sum(B * A) ** 2 / N)
            evs += C * np.sum(B * A)

        rel_err = np.sqrt(vars) / evs
        rows.append([i, r, f, rel_err])

In [ ]:
df = pd.DataFrame(rows, columns=['param', 'rig', 'flux', 'rel_err'])
df['param'] = df['param'].astype(int)
df['rig'] = df['rig'].astype(float)
df['flux'] = df['flux'].astype(float)
df['rel_err'] = df['rel_err'].astype(float)
df.sort_values(df.columns.to_list(), inplace=True)

In [ ]:
df2 = df.groupby(['rig']).agg(flux_mean=('flux', 'mean'),
                              flux_std=('flux', 'std'),
                              flux_rel_err=('flux', lambda x: x.std() / x.mean()),
                              est_rel_err=('rel_err', 'min'),
                              est_rel_err_var=('rel_err', 'var')).reset_index()

In [ ]:
(df2['flux_rel_err'] / df2['est_rel_err']).mean()

In [ ]:
np.sqrt(3)